# 🌲 تدريب نموذج YOLOv8 متعدد الفئات لرصد وعد أشجار المزارع والمصدات (Multi-Class Tree Crown Detection)
### متوافق تماماً مع مخطط المشروع الأساسي `dataset/data.yaml` (512px / 1024px · 50 Epochs)

- `0: orchard_tree` (أشجار البستان: موالح، مانجو، زيتون)
- `1: palm_tree` (نخيل)
- `2: windbreak_tree` (مصدات الرياح: كازورينا، كافور)

## 1️⃣ تثبيت المكتبات وفحص كارت الشاشة GPU

In [ ]:
!pip install -q ultralytics pyyaml scikit-learn
import torch
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print(f"CUDA Available: {torch.cuda.is_available()} | Selected device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

## 2️⃣ التحقق من ملف البيانات القياسي `dataset/data.yaml` وتوازن الفئات

In [ ]:
from pathlib import Path
import yaml
from train_yolo_tree_model import prepare_hard_negatives, oversample_and_copypaste_rare_classes

DATA_YAML = Path('dataset/data.yaml')
assert DATA_YAML.exists(), f'Missing canonical {DATA_YAML}!'

cfg = yaml.safe_load(DATA_YAML.read_text(encoding='utf-8'))
expected_names = {0: 'orchard_tree', 1: 'palm_tree', 2: 'windbreak_tree'}
assert cfg.get('names') == expected_names, f"Class mismatch in {DATA_YAML}: {cfg.get('names')}"

audit_stats = prepare_hard_negatives(DATA_YAML.resolve().parent)
oversample_and_copypaste_rare_classes(DATA_YAML.resolve().parent)
print('✅ Canonical dataset/data.yaml verified:', cfg)

## 3️⃣ بدء التدريب باستخدام `dataset/data.yaml` (512px أو 1024px للأشجار الصغيرة)

In [ ]:
from ultralytics import YOLO

BASE_MODEL = 'yolov8n.pt'  # Use 'yolov8s.pt' for dense 1024px training
IMGSZ = 512               # 512 matches SAHI 512px tiles; switch to 1024 for small crowns (8-25px)
EPOCHS = 50

model = YOLO(BASE_MODEL)
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=16,
    patience=12,
    save=True,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.4,
    degrees=180.0,
    flipud=0.5,
    fliplr=0.5,
    mosaic=0.8,
    device=DEVICE,
)
print('✅ اكتمل التدريب بنجاح!')

## 4️⃣ تصدير الأوزان النهائية `models/best.pt` و `public/models/best.onnx`

In [ ]:
import shutil
from pathlib import Path

save_dir = Path(model.trainer.save_dir)
best_weights = save_dir / 'weights' / 'best.pt'
Path('models').mkdir(parents=True, exist_ok=True)
Path('public/models').mkdir(parents=True, exist_ok=True)
shutil.copy2(best_weights, 'models/best.pt')
shutil.copy2(best_weights, 'best.pt')

best_model = YOLO(str(best_weights))
onnx_path = Path(best_model.export(format='onnx', imgsz=IMGSZ, opset=12, simplify=True, dynamic=False))
shutil.copy2(onnx_path, 'public/models/best.onnx')
shutil.copy2(onnx_path, 'models/best.onnx')
print('✅ Exported models/best.pt and public/models/best.onnx')